In [1]:
# Creath the entity for the training model 

from dataclasses import dataclass
from pathlib import Path
import os
import tensorflow as tf

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augumentation: bool
    params_image_size: list
    params_learning_rate: float

I0000 00:00:1780541965.255247   30431 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780541965.298048   30431 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780541966.206653   30431 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [3]:
class ConfigurationManager:
    def __init__(self, config_filepath=str(CONFIG_FILE_PATH), params_filepath=str(PARAMS_FILE_PATH)):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
        
    # From the config and params set the data members of TrainigConfig
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "kidney_ct_scan_dataset/kidney_ct_scan_dataset")
        create_directories([Path(training.root_dir)])

        training_config = TrainingConfig(root_dir=Path(training.root_dir), 
                                         trained_model_path=Path(training.trained_model_path),
                                         updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
                                         training_data=Path(training_data),
                                         params_epochs=params.EPOCHS,
                                         params_batch_size=params.BATCH_SIZE,
                                         params_is_augumentation=params.AUGUMENTATION,
                                         params_image_size=params.IMAGE_SIZE,
                                         params_learning_rate=params.LEARNING_RATE)

        return training_config

In [4]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path)
       

    def train_valid_generator(self):
    
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )
    
        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )
    
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
    
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )
    
        if self.config.params_is_augumentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator
    
        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)


    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=self.config.params_learning_rate),
                           loss=tf.keras.losses.CategoricalCrossentropy(),
                           metrics=['accuracy'])

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [6]:
try:
    config = ConfigurationManager()
    training_model_config = config.get_training_config()
    training = Training(training_model_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
except Exception as e:
    raise e

[2026-06-04 08:29:43,171: INFO: common: common.py: 26: Loaded config from /home/codebind/Documents/pet_projects/Kidney-disease-Classification-/config/config.yaml successfully!!!!]
[2026-06-04 08:29:43,194: INFO: common: common.py: 26: Loaded config from /home/codebind/Documents/pet_projects/Kidney-disease-Classification-/params.yaml successfully!!!!]
[2026-06-04 08:29:43,195: INFO: common: common.py: 45: Created directory at :artifacts]
[2026-06-04 08:29:43,195: INFO: common: common.py: 45: Created directory at :artifacts/training]


I0000 00:00:1780541985.481195   30431 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4166 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


[2026-06-04 08:29:46,594: WARNING: saving_utils: saving_utils.py: 249: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 2487 images belonging to 4 classes.
Found 9959 images belonging to 4 classes.


I0000 00:00:1780541987.097731   30431 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1780541987.653016   30508 service.cc:153] XLA service 0x74d464031bf0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780541987.653039   30508 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3050 6GB Laptop GPU, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.0)
I0000 00:00:1780541987.696981   30508 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1780541987.937832   30508 cuda_dnn.cc:461] Loaded cuDNN version 92300
I0000 00:00:1780541987.946118   30508 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1375__.68


  2/622 ━━━━━━━━━━━━━━━━━━━━ 57s 93ms/step - accuracy: 0.3281 - loss: 3.7596  

I0000 00:00:1780541994.751545   30508 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


152/622 ━━━━━━━━━━━━━━━━━━━━ 1:11 153ms/step - accuracy: 0.3243 - loss: 20.3998

I0000 00:00:1780542018.128543   30508 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1375__.68


622/622 ━━━━━━━━━━━━━━━━━━━━ 122s 184ms/step - accuracy: 0.4076 - loss: 14.2156 - val_accuracy: 0.5218 - val_loss: 8.9955
[2026-06-04 08:31:48,924: WARNING: saving_api: saving_api.py: 83: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]
